In [2]:
import pandas as pd
import boto3
import os
import logging
from botocore.exceptions import ClientError

### IMPORT

In [3]:
df_cities = pd.read_csv('assets/top_35_clean.csv', index_col=0)
df_coords = pd.read_csv('assets/top_35_coords_final.csv', index_col=0)
df_temp = pd.read_csv('assets/top_35_temperature.csv', index_col=0)

### JOIN

In [4]:
df_cities_coords = pd.merge(df_cities, df_coords, left_index=True, right_index=True)
df_cities_coords_temp = pd.merge(df_cities_coords, df_temp, left_index=True, right_index=True)
df_cities_coords_temp

,Cities,lat,lon,temp_K
0,Mont Saint Michel,48.6359541,-1.511459954959514,276.98
1,St Malo,48.649518,-2.0260409,276.86
2,Bayeux,49.2764624,-0.7024738,278.19
3,Le Havre,49.4938975,0.1079732,275.30
4,Rouen,49.4404591,1.0939658,274.74
5,Paris,48.8588897,2.3200410217200766,275.08
6,Amiens,49.8941708,2.2956951,273.35
7,Lille,50.6365654,3.0635282,273.65
8,Strasbourg,48.584614,7.7507127,269.36
9,Chateau du Haut Koenigsbourg,48.2495226,7.3454923,265.27


In [5]:
def celsious(column):
    return column + (-273)
df_cities_coords_temp['temp_C']= df_cities_coords_temp['temp_K'].agg(celsious)
df_cities_coords_temp

,Cities,lat,lon,temp_K,temp_C
0,Mont Saint Michel,48.6359541,-1.511459954959514,276.98,3.98
1,St Malo,48.649518,-2.0260409,276.86,3.86
2,Bayeux,49.2764624,-0.7024738,278.19,5.19
3,Le Havre,49.4938975,0.1079732,275.30,2.30
4,Rouen,49.4404591,1.0939658,274.74,1.74
5,Paris,48.8588897,2.3200410217200766,275.08,2.08
6,Amiens,49.8941708,2.2956951,273.35,0.35
7,Lille,50.6365654,3.0635282,273.65,0.65
8,Strasbourg,48.584614,7.7507127,269.36,-3.64
9,Chateau du Haut Koenigsbourg,48.2495226,7.3454923,265.27,-7.73


In [6]:
df_cities_coords_temp.to_csv('assets/df_cities_coords_temp.csv')

### AWS

In [7]:
AWS_KEY_ID = os.environ.get('AWS_KEY_ID')
AWS_SECRET = os.environ.get('AWS_SECRET')

In [8]:
# Generate the boto3 client for interacting with S3
s3 = boto3.client('s3', region_name='eu-west-3', 
                        # Set up AWS credentials 
                        aws_access_key_id=AWS_KEY_ID, 
                         aws_secret_access_key=AWS_SECRET)
s3


In [9]:
# List the buckets
buckets = s3.list_buckets()

# Print the buckets
print(type(buckets))
print(buckets)
print()
print(buckets['ResponseMetadata']['RequestId'])
print(buckets['ResponseMetadata']['HostId'])
print(buckets['ResponseMetadata']['HTTPStatusCode'])
print(buckets['ResponseMetadata']['HTTPHeaders'])
print(buckets['ResponseMetadata']['RetryAttempts'])
print()
for i in range(len(buckets['Buckets'])):
    print(buckets['Buckets'][i])

<class 'dict'>
{'ResponseMetadata': {'RequestId': '873Y5KBH45QJVSKR', 'HostId': '+hSopmwVhEe+2H1VNvENa9qptAFE1rUvTsAol1w5A9j9j/ZxK6Slvu2ErheUe3yiKqU0ZQ9J4+3cxiif5tNQ6g==', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': '+hSopmwVhEe+2H1VNvENa9qptAFE1rUvTsAol1w5A9j9j/ZxK6Slvu2ErheUe3yiKqU0ZQ9J4+3cxiif5tNQ6g==', 'x-amz-request-id': '873Y5KBH45QJVSKR', 'date': 'Thu, 13 Jan 2022 10:55:37 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'javipsanchez2', 'CreationDate': datetime.datetime(2021, 10, 3, 20, 49, 31, tzinfo=tzutc())}, {'Name': 'mlflow-sagemaker-eu-west-3-237610266389', 'CreationDate': datetime.datetime(2021, 11, 24, 15, 13, 7, tzinfo=tzutc())}, {'Name': 'scoring-cities-in-the-world-exemple', 'CreationDate': datetime.datetime(2021, 10, 3, 21, 19, 20, tzinfo=tzutc())}], 'Owner': {'ID': '23a523f7dc3ef87cdb277af7dde64bd622c4ca4f1c64bb215bf5879eeaed3763'}}

873Y5KBH45QJVSKR
+hSopmwVhEe+2H1VNv

In [10]:
# Retrieve the list of existing buckets
s3 = boto3.client('s3')
response = s3.list_buckets()

# Output the bucket names
print('Existing buckets:')
for bucket in response['Buckets']:
    print(f'  {bucket["Name"]}')

Existing buckets:
  javipsanchez2
  mlflow-sagemaker-eu-west-3-237610266389
  scoring-cities-in-the-world-exemple


In [11]:
def create_bucket(bucket_name, region):
    """Create an S3 bucket in a specified region

    If a region is not specified, the bucket is created in the S3 default
    region (us-east-1).

    :param bucket_name: Bucket to create
    :param region: String region to create bucket in, e.g., 'us-west-2'
    :return: True if bucket created, else False
    """

    # Create bucket
    try:
        if region is None:
            s3_client = boto3.client('s3')
            s3_client.create_bucket(Bucket=bucket_name)
        else:
            s3_client = boto3.client('s3', region_name=region)
            location = {'LocationConstraint': region}
            s3_client.create_bucket(Bucket=bucket_name,
                                    CreateBucketConfiguration=location)
    except ClientError as e:
        logging.error(e)
        return False
    return True

In [13]:
create_bucket("plan-your-trip-kayak", 'eu-west-3')

True

In [15]:
# Upload final_report.csv to gid-staging
s3.upload_file(Bucket='plan-your-trip-kayak',
              # Set filename and key
               Filename='assets/df_cities_coords_temp.csv', 
               Key='df_cities_coords_temp.csv')

In [17]:
# Get object metadata and print it
response = s3.head_object(Bucket='plan-your-trip-kayak', 
                       Key='df_cities_coords_temp.csv')
response

{'ResponseMetadata': {'RequestId': '3MTNQ5N6YH4K5V7X',
  'HostId': '2SA+qHCfB4tFELXiSVp6fo9FK4lgpDTQmPRen6N3hbthsi2HBeUwwoZiIDxWCLi3eDhxEW4NwUs=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '2SA+qHCfB4tFELXiSVp6fo9FK4lgpDTQmPRen6N3hbthsi2HBeUwwoZiIDxWCLi3eDhxEW4NwUs=',
   'x-amz-request-id': '3MTNQ5N6YH4K5V7X',
   'date': 'Thu, 13 Jan 2022 10:58:54 GMT',
   'last-modified': 'Thu, 13 Jan 2022 10:58:38 GMT',
   'etag': '"552ce88276920b4702c4aae20144dfac"',
   'accept-ranges': 'bytes',
   'content-type': 'binary/octet-stream',
   'server': 'AmazonS3',
   'content-length': '1084'},
  'RetryAttempts': 0},
 'AcceptRanges': 'bytes',
 'LastModified': datetime.datetime(2022, 1, 13, 10, 58, 38, tzinfo=tzutc()),
 'ContentLength': 1084,
 'ETag': '"552ce88276920b4702c4aae20144dfac"',
 'ContentType': 'binary/octet-stream',
 'Metadata': {}}

In [18]:
# Print the size of the uploaded object
print(response['ContentLength'])

1084
